### Pytorch Coding of the Flash Attention (SDPA) - Scaled dot product attention

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class FlashAttention(nn.Module):

    def __init__(self, embed_dim, num_heads, dropout=0.0):

        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)

        self.dropout = dropout


    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # PyTorch's optimized attention (similar to FlashAttention)
        out = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True
        )

        out = out.transpose(1, 2).reshape(B, T, C)

        return self.out_proj(out)

# 🚀 Flash Attention (PyTorch SDPA) – Step-by-Step Explanation

Your code implements **Multi-Head Attention using PyTorch’s optimized

`scaled_dot_product_attention` kernel**, which internally behaves like 

**FlashAttention** (efficient GPU attention).

---

### 1️⃣ Importing Required Libraries

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
```

### What each import does

| Library               | Purpose                                         |
| --------------------- | ----------------------------------------------- |
| `torch`               | Core tensor operations                          |
| `torch.nn`            | Neural network modules                          |
| `torch.nn.functional` | Low-level functions like attention, activations |

The important function used later is:

```
F.scaled_dot_product_attention()
```

This is **PyTorch's fused attention kernel** which is:

* memory efficient
* GPU optimized
* uses FlashAttention internally when possible

---

### 🧠 FlashAttention Class

---

### 2️⃣ Defining the Attention Module

```python
class FlashAttention(nn.Module):
```

This creates a **custom neural network layer** for attention.

It inherits from:

```
nn.Module
```

Which allows the module to:

* store parameters
* integrate into PyTorch models
* support `.forward()` execution

---

### ⚙️ Initialization

---

### 3️⃣ Constructor (`__init__`)

```python
def __init__(self, embed_dim, num_heads, dropout=0.0):
```

### Parameters

| Parameter   | Meaning                   |
| ----------- | ------------------------- |
| `embed_dim` | token embedding size      |
| `num_heads` | number of attention heads |
| `dropout`   | regularization            |

Example:

```
embed_dim = 768
num_heads = 12
```

---

### 4️⃣ Calling Parent Constructor

```python
super().__init__()
```

This initializes the **base `nn.Module` class**.

Without this, PyTorch cannot track parameters.

---

### 5️⃣ Saving Model Hyperparameters

```python
self.embed_dim = embed_dim
self.num_heads = num_heads
self.head_dim = embed_dim // num_heads
```

### Head dimension formula

$$
d_{head} = \frac{d_{model}}{n_{heads}}
$$

Where

* $d_{model}$ = embedding dimension
* $n_{heads}$ = number of attention heads

Example

```
embed_dim = 768
num_heads = 12

head_dim = 64
```

---

### 🔧 Linear Projection Layers

---

### 6️⃣ Query Projection

```python
self.q_proj = nn.Linear(embed_dim, embed_dim)
```

Transforms input tokens into **Query vectors**.

Formula:

$$
Q = XW_Q
$$

Where

* $X$ = input embeddings
* $W_Q$ = query weights

Shape:

```
[B, T, C] → [B, T, C]
```

---

### 7️⃣ Key Projection

```python
self.k_proj = nn.Linear(embed_dim, embed_dim)
```

Formula:

$$
K = XW_K
$$

Used to measure similarity with queries.

---

### 8️⃣ Value Projection

```python
self.v_proj = nn.Linear(embed_dim, embed_dim)
```

Formula:

$$
V = XW_V
$$

Values contain the **information to be aggregated**.

---

### 9️⃣ Output Projection

```python
self.out_proj = nn.Linear(embed_dim, embed_dim)
```

After attention across heads, outputs are merged and projected.

Formula

$$
Output = Concat(head_1,...,head_h)W_O
$$

---

### 🔟 Dropout

```python
self.dropout = dropout
```

Dropout is applied **inside attention weights**.

Purpose:

* reduce overfitting
* improve generalization

---

### ⚡ Forward Pass

---

### 1️⃣1️⃣ Input Tensor

```python
B, T, C = x.shape
```

Input shape:

```
x → [B, T, C]
```

Where

| Symbol | Meaning             |
| ------ | ------------------- |
| B      | batch size          |
| T      | sequence length     |
| C      | embedding dimension |

Example

```
B = 8
T = 512
C = 768
```

---

### 🔎 Compute Q, K, V

---

### 1️⃣2️⃣ Query

```python
q = self.q_proj(x)
```

Shape:

```
[B, T, C]
```

---

### 1️⃣3️⃣ Key

```python
k = self.k_proj(x)
```

Shape:

```
[B, T, C]
```

---

### 1️⃣4️⃣ Value

```python
v = self.v_proj(x)
```

Shape:

```
[B, T, C]
```

---

### 🧩 Split into Multiple Heads

---

### 1️⃣5️⃣ Reshape Queries

```python
q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
```

First reshape

```
[B, T, C]
↓
[B, T, H, D]
```

Where

* H = heads
* D = head_dim

Then transpose:

```
[B, H, T, D]
```

Final shape:

```
q → [B, H, T, D]
```

---

### 1️⃣6️⃣ Reshape Keys

```python
k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
```

Final shape:

```
k → [B, H, T, D]
```

---

### 1️⃣7️⃣ Reshape Values

```python
v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
```

Final shape:

```
v → [B, H, T, D]
```

---

### ⚡ Flash Attention Kernel

---

### 1️⃣8️⃣ Scaled Dot Product Attention

```python
out = F.scaled_dot_product_attention(
    q, k, v,
    dropout_p=self.dropout if self.training else 0.0,
    is_causal=True
)
```

This function performs **attention in a single optimized GPU kernel**.

---

### Attention Formula

$$
Attention(Q,K,V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Where

* $Q$ = queries
* $K$ = keys
* $V$ = values
* $d_k$ = key dimension

---

### Step-by-step internally

1️⃣ Compute similarity

$$
S = QK^T
$$

Shape:

```
[B, H, T, T]
```

---

2️⃣ Scale scores

$$
S = \frac{S}{\sqrt{d_k}}
$$

Scaling prevents softmax instability.

---

3️⃣ Apply causal mask

Because

```
is_causal = True
```

Future tokens are masked.

Mask example:

```
T0 ■
T1 ■ ■
T2 ■ ■ ■
T3 ■ ■ ■ ■
```

---

4️⃣ Softmax

$$
A = softmax(S)
$$

Attention weights.

---

5️⃣ Multiply with values

$$
Output = AV
$$

Shape

```
[B, H, T, D]
```

---

### 🔁 Merge Heads

---

### 1️⃣9️⃣ Recombine Attention Heads

```python
out = out.transpose(1, 2).reshape(B, T, C)
```

Step 1:

```
[B, H, T, D]
↓
[B, T, H, D]
```

Step 2 reshape

```
[B, T, H×D]
```

Since

```
H × D = C
```

Final:

```
[B, T, C]
```

---

### 🧱 Final Output Projection

---

### 2️⃣0️⃣ Linear Projection

```python
return self.out_proj(out)
```

Formula

$$
Y = OutW_O
$$

Shape remains

```
[B, T, C]
```

This mixes information from all heads.

---

### 📊 Complete Tensor Shape Flow

```
Input
[B, T, C]

Q,K,V projection
[B, T, C]

Split heads
[B, H, T, D]

Flash Attention
[B, H, T, D]

Merge heads
[B, T, C]

Output projection
[B, T, C]
```

---

### ⚡ Why This Is Efficient

Using

```
F.scaled_dot_product_attention()
```

gives major improvements.

### 1️⃣ FlashAttention kernel

* avoids storing large attention matrix
* reduces memory usage

---

### 2️⃣ GPU fused operations

Computes

```
QKᵀ
Softmax
AV
```

in **one kernel**.

---

### 3️⃣ Memory Complexity

Standard attention:

$$
O(n^2)
$$

FlashAttention memory:

$$
O(n)
$$

Huge improvement for long sequences.

---

### 🏆 Summary

This code implements:

**Causal Multi-Head Attention using PyTorch FlashAttention kernel**

Pipeline:

```
Input → QKV projection
     → Split heads
     → FlashAttention kernel
     → Merge heads
     → Output projection
```

Benefits:

✅ Faster training
✅ Lower GPU memory
✅ Built-in causal masking
✅ Production ready

---
